# Jimmy trainieren auf Google Colab (gratis GPU)
Laufzeit > Laufzeittyp ändern > **GPU (T4)**. Free-Sitzungen enden nach einigen Stunden: trainiere **einen Task pro Sitzung** und speichere Checkpoints auf Google Drive.

In [ ]:
!nvidia-smi
!pip -q install uv
!git clone https://github.com/LennardOwnTest123006/microduck_rl_learn-Jimmy
%cd microduck_rl_learn-Jimmy
!uv sync

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Smoke-Test (immer zuerst)
!uv run train Mjlab-Velocity-Flat-MicroDuck --env.scene.num-envs 64 --agent.max_iterations 5

In [ ]:
# Echtes Training: TASK ändern (siehe docs/JIMMY_ANLEITUNG.md). Weniger Envs als 4096, damit die T4 reicht.
TASK='Mjlab-Velocity-Flat-MicroDuck'
!uv run train $TASK --env.scene.num-envs 2048 --agent.max_iterations 3000

In [ ]:
# Logs/Checkpoints sichern
!cp -r logs /content/drive/MyDrive/jimmy_logs

In [ ]:
# ONNX exportieren (Pfad aus logs/ oder wandb) und veröffentlichen, siehe Anleitung Schritt 3
# !uv run scripts/export.py $TASK --wandb-run-path <entity/project/run_id>

## Zuschauen
**Live-Kurven:** vor dem Training einmal `wandb login` (kostenloses Konto auf wandb.ai), dann siehst du auf wandb.ai live, wie Reward und Episodenlänge steigen.
**Video:** nach dem Training (oder zwischendurch, wenn schon ein Checkpoint existiert) das Video unten erzeugen.

In [ ]:
!uv run wandb login

In [ ]:
# Video des trainierten Jimmy rendern (unerprobt: Flags kommen aus der play-Konfiguration des Repos)
import glob
ckpt = sorted(glob.glob('logs/**/model_*.pt', recursive=True), key=lambda p:int(p.split('model_')[1].split('.')[0]))[-1]
print('Checkpoint:', ckpt)
!uv run play $TASK --checkpoint-file $ckpt --num-envs 1 --video True --video-length 400

In [ ]:
# Video anzeigen
import glob
from IPython.display import HTML, display
from base64 import b64encode
v = sorted(glob.glob('**/*.mp4', recursive=True))[-1]
print(v)
display(HTML('<video width=640 controls src="data:video/mp4;base64,%s"></video>' % b64encode(open(v,'rb').read()).decode()))